# Encapsulation, Composition, and TDD with Pytest (Solution)

**Duration:** 2 hours

**Objectives -**
- Refactor Week 2 code to enforce encapsulation
- Use composition instead of inheritance where applicable
- Understand trade-offs between inheritance and composition
- Add Pythonic @property accessors
- Introduce unit testing with `pytest`
- Design new functionality driven by failing tests
- Practice mocking and interface-driven design

## Section 0: Setup

In [ ]:
!pip install pytest pytest-mock

## Section 1: Recap and Refactor

- Enforce encapsulation and proper object structure in Week 2’s code
- Refactor `Account` and related classes to use private attributes, properties, and composition.

In [ ]:
class TransactionLogger:
    def log(self, message):
        print(f"LOG: {message}")

class NotificationService:
    def notify(self, user, message):
        print(f"Notify {user}: {message}")

class Account:
    def __init__(self, owner, balance, logger=None, notifier=None):
        self._owner = owner
        self._balance = balance
        self._logger = logger or TransactionLogger()
        self._notifier = notifier or NotificationService()

    @property
    def owner(self):
        return self._owner

    @property
    def balance(self):
        return self._balance

    def deposit(self, amount):
        if amount > 0:
            self._balance += amount
            self._logger.log(f"Deposited {amount} to {self._owner}")
            self._notifier.notify(self._owner, f"Deposit: {amount}")

    def withdraw(self, amount):
        if 0 < amount <= self._balance:
            self._balance -= amount
            self._logger.log(f"Withdrew {amount} from {self._owner}")
            self._notifier.notify(self._owner, f"Withdraw: {amount}")
            return True
        else:
            self._logger.log(f"Failed withdrawal for {self._owner}")
            return False

## Section 2: Properties in Practice

- Add at least 2 @property methods and 1 computed property in the refactored classes.

In [ ]:
class AccountWithFee(Account):
    def __init__(self, owner, balance, monthly_fee, logger=None, notifier=None):
        super().__init__(owner, balance, logger, notifier)
        self._monthly_fee = monthly_fee

    @property
    def monthly_fee(self):
        return self._monthly_fee

    @property
    def net_balance(self):
        """Computed property: balance after fee."""
        return self.balance - self.monthly_fee

acc = AccountWithFee("Alice", 200, 10)
print("Owner:", acc.owner)
print("Balance:", acc.balance)
print("Monthly Fee:", acc.monthly_fee)
print("Net Balance:", acc.net_balance)

## Section 3: Composition vs Inheritance

- Example: Refactor an inherited class to use composition instead.

In [ ]:
# Instead of: class RewardsAccount(Account)
class RewardSystem:
    def __init__(self):
        self.points = 0
    def add_points(self, amount):
        self.points += int(amount // 10)

class AccountWithReward:
    def __init__(self, account, reward_system):
        self._account = account
        self._reward_system = reward_system

    @property
    def balance(self):
        return self._account.balance

    def deposit(self, amount):
        self._account.deposit(amount)
        self._reward_system.add_points(amount)

    @property
    def reward_points(self):
        return self._reward_system.points

# Usage
base_acc = Account("Bob", 100)
rewards = RewardSystem()
acc_with_reward = AccountWithReward(base_acc, rewards)
acc_with_reward.deposit(50)
print("Balance:", acc_with_reward.balance)
print("Reward Points:", acc_with_reward.reward_points)

**Explanation:**

Composition allows us to add features (like rewards) without modifying or tightly coupling to the base `Account` class. This is more flexible than inheritance when features are optional or orthogonal.

## Section 4: Unit Testing with Pytest

In [ ]:
import pytest

class TestAccount:
    def setup_method(self):
        self.acc = Account("TestUser", 100)

    def teardown_method(self):
        del self.acc

    def test_balance_property(self):
        assert self.acc.balance == 100

    def test_deposit(self):
        self.acc.deposit(50)
        assert self.acc.balance == 150

    def test_withdraw(self):
        result = self.acc.withdraw(30)
        assert result is True
        assert self.acc.balance == 70

    def test_withdraw_insufficient(self):
        result = self.acc.withdraw(200)
        assert result is False
        assert self.acc.balance == 100

## Section 5: TDD with New Feature

- Design a new class `TransactionService` that:
    - Performs a transaction between two accounts
    - Logs the event
    - Raises errors if balance is insufficient

In [ ]:
import pytest

class TransactionService:
    def __init__(self, logger=None):
        self._logger = logger or TransactionLogger()

    def transfer(self, from_acc, to_acc, amount):
        if from_acc.balance < amount:
            self._logger.log("Transfer failed: insufficient funds")
            raise ValueError("Insufficient funds")
        from_acc.withdraw(amount)
        to_acc.deposit(amount)
        self._logger.log(f"Transferred {amount} from {from_acc.owner} to {to_acc.owner}")

def test_transfer_fails_on_low_balance():
    acc1 = Account("A", 20)
    acc2 = Account("B", 50)
    svc = TransactionService()
    with pytest.raises(ValueError):
        svc.transfer(acc1, acc2, 100)

def test_transfer_success():
    acc1 = Account("A", 100)
    acc2 = Account("B", 50)
    svc = TransactionService()
    svc.transfer(acc1, acc2, 40)
    assert acc1.balance == 60
    assert acc2.balance == 90

## Section 6: Mocking Collaborators

In [ ]:
def test_logger_called(mocker):
    mock_logger = mocker.Mock()
    acc1 = Account("A", 100)
    acc2 = Account("B", 50)
    svc = TransactionService(logger=mock_logger)
    svc.transfer(acc1, acc2, 10)
    mock_logger.log.assert_any_call('Transferred 10 from A to B')

## Section 7: Reflection Questions

- **When would you choose composition over inheritance?**
    - When you want to add features without tightly coupling or modifying the base class. Composition is more flexible for optional or cross-cutting concerns.
- **What are benefits of @property?**
    - Encapsulates internal state, allows computed or read-only attributes, and provides a clean API.
- **What did mocking help simplify?**
    - Mocking allows us to test interactions and side effects (like logging) without relying on real implementations.
- **What did you find hardest?**
    - Refactoring for composition and designing for testability can be challenging at first.

## Additional Section

Add interface enforcement using `abc.ABC`, e.g.:

In [ ]:
from abc import ABC, abstractmethod

class Logger(ABC):
    @abstractmethod
    def log(self, message: str):
        pass

class ConsoleLogger(Logger):
    def log(self, message: str):
        print(f"[LOG] {message}")